# LAB 2 — Contact-Consistent Fixed-Base Dynamics

## Learning objectives

You will:

- generate a sinusoidal joint-space reference;
- compute dynamically consistent contact projection quantities;
- project velocities at an inelastic impact;
- simulate constrained and unconstrained forward dynamics;
- verify that contact forces disappear after projection;
- generate a constraint-consistent joint reference;
- verify Gauss's principle through a quadratic program;
- change the contact location;
- optionally verify the spatial shifting law.

> **Environment.** This lab assumes Pinocchio, `base_controllers`, ROS/RViz support used by `RosPub`, and the quadratic-program helper `quadprog_solve_qp`.

## Contact-consistent dynamics

For a 3D point contact with Jacobian \(J\), the supplied lab uses

$
\ddot q_c =
M^{-1}
\left[
  N_{\bar M}(S^T\tau-h)
  -J^T\Lambda\,\dot J\dot q
\right],
$

where

$
\Lambda=(J M^{-1}J^T)^{-1},
$

and $N_{\bar M}$ projects generalized forces into the dynamically consistent null space of the contact.

The contact acceleration constraint is

$
J\ddot q + \dot J\dot q = 0.
$

## 0. Imports and self-contained configuration

In [1]:
import pinocchio as pin
from pinocchio.utils import *
import numpy as np
from numpy import nan
import math
import time as tm
import os

from base_controllers.utils.common_functions import *
from base_controllers.utils.optimTools import quadprog_solve_qp
from base_controllers.utils.ros_publish import RosPub

# Configuration copied from the supplied L5_conf.py
dt = 0.001
exp_duration_sin = 3.0
exp_duration = 5.0
SLOW_FACTOR = 1

frame_name = "tool0"

kp = np.eye(6) * 200.0
kd = np.eye(6) * 20.0

amp = np.array([0.0, 0.6, 0.0, 0.0, 0.0, 0.0])
phi = np.zeros(6)
freq = np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0])

q0 = np.array([0.0, -0.6, 0.6, -1.67, -1.57, 0.0])
qd0 = np.zeros(6)
qdd0 = np.zeros(6)

np.set_printoptions(precision=4, suppress=True, linewidth=160)

## 1. Sinusoidal Shoulder-Lift reference

The lab specifies amplitude \(0.6\,\mathrm{rad}\) and frequency \(1\,\mathrm{Hz}\) for the Shoulder Lift joint.

For each joint,

$
q^d(t)=q_0 + a\sin(2\pi f t+\phi),
$

$
\dot q^d(t)=2\pi f\,a\cos(2\pi f t+\phi),
$

$
\ddot q^d(t)=-(2\pi f)^2a\sin(2\pi f t+\phi).
$

In [2]:
two_pi_f = 2.0 * np.pi * freq
two_pi_f_amp = two_pi_f * amp
two_pi_f_squared_amp = two_pi_f * two_pi_f_amp

def sinusoidal_reference(t, q_center=q0, amplitude=amp):
    q_des = q_center + amplitude * np.sin(two_pi_f * t + phi)
    qd_des = (two_pi_f * amplitude) * np.cos(two_pi_f * t + phi)
    qdd_des = -(two_pi_f**2 * amplitude) * np.sin(two_pi_f * t + phi)
    return q_des, qd_des, qdd_des

for t in [0.0, 0.25, 0.5]:
    q_des, qd_des, qdd_des = sinusoidal_reference(t)
    print(f"t={t:4.2f} s | q_des={q_des}")

t=0.00 s | q_des=[ 0.   -0.6   0.6  -1.67 -1.57  0.  ]
t=0.25 s | q_des=[ 0.    0.    0.6  -1.67 -1.57  0.  ]
t=0.50 s | q_des=[ 0.   -0.6   0.6  -1.67 -1.57  0.  ]


## 2. Constraint-consistent quantities

The most direct way to compute \(\dot J\dot q\) in this lab is the one suggested in the description: compute the classical linear acceleration of the contact frame with \(\ddot q=0\).

The supplied implementation uses the dynamically consistent inverse of \(J^T\),

$
J^{T\#} = \Lambda J M^{-1},
$

and

$
N_{\bar M}=I-J^T J^{T\#}.
$

In [3]:
print(os.environ["LOCOSIM_DIR"]+"/robot_descriptions/ur_description/urdf/ur5.urdf.xacro")
robot = getRobotModel(robot_name="ur5", generate_urdf=True, xacro_path=os.environ["LOCOSIM_DIR"]+"/robot_descriptions/ur_description/urdf/ur5.urdf.xacro")

assert robot.model.existFrame(frame_name)
frame_contact = robot.model.getFrameId(frame_name)

os.system("killall rosmaster >/dev/null 2>&1")
ros_pub = RosPub("ur5")

def contact_quantities(robot, q, qd, frame_id):
    robot.computeAllTerms(q, qd)

    M = robot.mass(q, False)
    h = robot.nle(q, qd, False)
    g = robot.gravity(q)

    J6 = robot.frameJacobian(q, frame_id, False, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
    J = J6[:3, :]

    # Classical acceleration with qdd = 0 gives Jdot*qdot.
    dJdq = robot.frameClassicAcceleration(q, qd, None, frame_id).linear

    x = robot.framePlacement(q, frame_id).translation
    v_frame = robot.frameVelocity(q, qd, frame_id, False)
    xd = v_frame.linear

    M_inv = np.linalg.inv(M)
    Lambda = np.linalg.inv(J @ M_inv @ J.T)
    JT_dagger = Lambda @ J @ M_inv
    N_m = np.eye(robot.model.nv) - J.T @ JT_dagger

    return {
        "M": M, "M_inv": M_inv, "h": h, "g": g,
        "J6": J6, "J": J, "dJdq": dJdq,
        "x": x, "xd": xd,
        "Lambda": Lambda,
        "JT_dagger": JT_dagger,
        "N_m": N_m,
    }

cq = contact_quantities(robot, q0.copy(), qd0.copy(), frame_contact)
print("J shape:", cq["J"].shape)
print("Lambda:\n", cq["Lambda"])
print("||N_m J^T|| =", np.linalg.norm(cq["N_m"] @ cq["J"].T))

/root/ros_ws/src/wheresim/robot_descriptions/ur_description/urdf/ur5.urdf.xacro



URDF generated_commons
URDF loaded in Pinocchio
Starting ros pub---------------------------------------------------------------
... logging to /root/.ros/log/4dc2b118-b537-11f1-9e7b-5254007b7617/roslaunch-docker-9504.log
started roslaunch server http://docker:43367/

SUMMARY

PARAMETERS
 * /robot_description: <?xml version="1....
 * /rosdistro: noetic
 * /rosversion: 1.17.4

NODES
  /
    robot_state_publisher (robot_state_publisher/robot_state_publisher)
    rviz (rviz/rviz)

ROS_MASTER_URI=http://localhost:11311
process[rviz-1]: started with pid [9655]
process[robot_state_publisher-2]: started with pid [9656]
[INFO] [1789939069.199180]: RVIZ started
Initialized ros pub---------------------------------------------------------------
J shape: (3, 6)
Lambda:
 [[1.7389 0.0026 1.2907]
 [0.0026 1.0052 0.0053]
 [1.2907 0.0053 4.504 ]]
||N_m J^T|| = 2.893936984571763e-16


MESA: warning: Driver does not support the 0x7d55 PCI ID.
libGL error: failed to create dri screen
libGL error: failed to load driver: iris
MESA: warning: Driver does not support the 0x7d55 PCI ID.
libGL error: failed to create dri screen
libGL error: failed to load driver: iris


Error in XmlRpcClient::writeRequest: write error (Connection refused).


MESA: warning: Driver does not support the 0x7d55 PCI ID.
libGL error: failed to create dri screen
libGL error: failed to load driver: iris
MESA: warning: Driver does not support the 0x7d55 PCI ID.
libGL error: failed to create dri screen
libGL error: failed to load driver: iris


## 3. Inelastic impact: velocity projection

At touchdown the desired post-impact contact-point velocity is zero. The supplied exercise asks you to obtain a compatible joint velocity using a null-space projector for \(J\):

$
\dot q^+ = N\,\dot q^-,
\qquad
N=I-J^\#J.
$

The original script uses the Moore–Penrose right inverse

$
J^\#=J^T(JJ^T)^{-1}
$

for this *velocity* projection.

In [4]:
def project_velocity_after_impact(J, qd_minus):
    J_pinv = J.T @ np.linalg.inv(J @ J.T)
    N = np.eye(J.shape[1]) - J_pinv @ J
    qd_plus = N @ qd_minus
    return qd_plus, N

# Example with a nonzero pre-impact velocity
qd_minus = np.array([0.1, -0.5, 0.2, 0.1, -0.1, 0.2])
cq = contact_quantities(robot, q0.copy(), qd_minus.copy(), frame_contact)
qd_plus, N = project_velocity_after_impact(cq["J"], qd_minus)

print("Pre-impact contact velocity:", cq["J"] @ qd_minus)
print("Post-impact contact velocity:", cq["J"] @ qd_plus)

Pre-impact contact velocity: [ 0.0916 -0.0644  0.3172]
Post-impact contact velocity: [-0. -0. -0.]


## 4. Simulation of constrained dynamics

The cell below is a notebook-friendly version of the supplied control loop. It preserves the original PD + gravity controller and contact logic, but exposes exercise options as arguments.

A small implementation detail has been made explicit for notebook robustness: the contact force is initialized to zero before the loop so that the lift-off test never references it before assignment.

In [5]:
def run_contact_simulation(robot,
    contact_frame_name="tool0",
    amplitude=0.6,
    use_constraint_consistent_reference=False,
    verify_projection=False,
    realtime=True,
):

    frame_contact = robot.model.getFrameId(contact_frame_name)
    zero = np.zeros(6)
    q = q0.copy()
    qd = qd0.copy()
    qdd = qdd0.copy()
    f = np.zeros(3)

    amplitude_vec = np.array([0.0, amplitude, 0.0, 0.0, 0.0, 0.0])
    omega = 2.0 * np.pi * freq

    buffer_size = int(math.floor(exp_duration / dt))
    q_log = np.full((6, buffer_size), np.nan)
    q_des_log = np.full((6, buffer_size), np.nan)
    qd_log = np.full((6, buffer_size), np.nan)
    qd_des_log = np.full((6, buffer_size), np.nan)
    qdd_log = np.full((6, buffer_size), np.nan)
    qdd_des_log = np.full((6, buffer_size), np.nan)
    tau_log = np.full((6, buffer_size), np.nan)
    f_log = np.full((3, buffer_size), np.nan)
    x_log = np.full((3, buffer_size), np.nan)
    time_log = np.full(buffer_size, np.nan)

    z_gnd = 0.0
    ground_contact = False
    counter_lo = 0
    counter_td = 0
    N_vel = np.eye(6)

    q_des = q0.copy()
    time = 0.0
    k = 0

    while time < exp_duration:
        qd_des_raw = (omega * amplitude_vec) * np.cos(omega * time + phi)
        qdd_des_raw = -(omega**2 * amplitude_vec) * np.sin(omega * time + phi)

        if use_constraint_consistent_reference:
            qd_des = qd_des_raw.copy()
            qdd_des = qdd_des_raw.copy()
            if ground_contact:
                qd_des = N_vel @ qd_des
                qdd_des = N_vel @ qdd_des
            q_des = q_des + qd_des * dt
        else:
            q_des = q0 + amplitude_vec * np.sin(omega * time + phi)
            qd_des = qd_des_raw
            qdd_des = qdd_des_raw

        if time >= exp_duration_sin:
            q_des = q0.copy()
            qd_des = zero.copy()
            qdd_des = zero.copy()

        cq = contact_quantities(robot, q, qd, frame_contact)
        M, M_inv, h, g = cq["M"], cq["M_inv"], cq["h"], cq["g"]
        J, dJdq, x = cq["J"], cq["dJdq"], cq["x"]
        Lambda, N_m = cq["Lambda"], cq["N_m"]

        tau = kp @ (q_des - q) + kd @ (qd_des - qd) + g

        # Touchdown
        if counter_lo == 0 and (not ground_contact) and x[2] <= z_gnd:
            counter_td = 30
            ground_contact = True
            qd, N_vel = project_velocity_after_impact(J, qd)

            # Recompute velocity-dependent terms after the jump in qd.
            cq = contact_quantities(robot, q, qd, frame_contact)
            M, M_inv, h, g = cq["M"], cq["M_inv"], cq["h"], cq["g"]
            J, dJdq, x = cq["J"], cq["dJdq"], cq["x"]
            Lambda, N_m = cq["Lambda"], cq["N_m"]
            print(f"contact ON  at t={time:.3f} s")

        # Lift-off, with hysteresis
        if counter_td == 0 and ground_contact and f[2] <= 0.0:
            counter_lo = 200
            ground_contact = False
            print(f"contact OFF at t={time:.3f} s")

        counter_td = max(counter_td - 1, 0)
        counter_lo = max(counter_lo - 1, 0)

        qdd_uc = M_inv @ (tau - h)

        if not ground_contact:
            qdd = qdd_uc
            f = np.zeros(3)
        else:
            qdd = M_inv @ (
                N_m @ (tau - h) - J.T @ Lambda @ dJdq
            )
            f = Lambda @ (-dJdq + J @ M_inv @ (h - tau))

            if verify_projection and (k % 100 == 0):
                print("||N_m J^T f|| =", np.linalg.norm(N_m @ J.T @ f))


        # Integration, kept consistent with the supplied script.
        qd = qd + qdd * dt
        q = q + qd * dt + 0.5 * dt * dt * qdd

        time_log[k] = time
        q_log[:, k] = q
        q_des_log[:, k] = q_des
        qd_log[:, k] = qd
        qd_des_log[:, k] = qd_des
        qdd_log[:, k] = qdd
        qdd_des_log[:, k] = qdd_des
        tau_log[:, k] = tau
        f_log[:, k] = f
        x_log[:, k] = x

        if ros_pub is not None:
            ros_pub.add_arrow(x, f / 100.0)
            ros_pub.add_marker(x)
            ros_pub.publish(robot, q, qd, tau)

            if realtime:
                tm.sleep(dt * SLOW_FACTOR)

            if ros_pub.isShuttingDown():
                print("Shutting down")
                k += 1
                break

        time += dt
        k += 1

    if ros_pub is not None:
        ros_pub.deregister_node()

    sl = slice(0, k)
    return {
        "time": time_log[sl],
        "q": q_log[:, sl],
        "q_des": q_des_log[:, sl],
        "qd": qd_log[:, sl],
        "qd_des": qd_des_log[:, sl],
        "qdd": qdd_log[:, sl],
        "qdd_des": qdd_des_log[:, sl],
        "tau": tau_log[:, sl],
        "f": f_log[:, sl],
        "x": x_log[:, sl],
    }

### Run the baseline experiment

Start with the exact lab parameters. If you do not have ROS/RViz available in the notebook kernel, set `use_ros=False`.

In [6]:
logs = run_contact_simulation(robot,
    contact_frame_name="tool0",
    amplitude=0.6,
    verify_projection=False,
    use_constraint_consistent_reference=False,
)

contact ON  at t=0.169 s
contact OFF at t=0.341 s
contact ON  at t=1.182 s
contact OFF at t=1.330 s
contact ON  at t=2.182 s
contact OFF at t=2.331 s
contact ON  at t=3.282 s
contact OFF at t=3.312 s
---------------------------------------------------------------


## 5. Verify that projected contact forces disappear

During contact, check

$
N_{\bar M}J^Tf \approx 0.
$



In [7]:
# Set verify_projection=True to print the residual during contact:

## 6. Compare dynamically consistent and Moore–Penrose force projectors

The lab also asks you to compare the dynamically consistent projector with one built from the ordinary Moore–Penrose inverse. The latter does not, in general, provide the same force-space orthogonality with respect to the inertia metric.

In [8]:
q_test = q0.copy()
qd_test = np.zeros(6)
cq = contact_quantities(robot, q_test, qd_test, frame_contact)

J = cq["J"]
M_inv = cq["M_inv"]

# Dynamically consistent projector from the lab.
Lambda = np.linalg.inv(J @ M_inv @ J.T)
JT_dyn = Lambda @ J @ M_inv
N_dyn = np.eye(6) - J.T @ JT_dyn

# Moore-Penrose construction for J^T.
JT_mp = np.linalg.pinv(J.T)
N_mp = np.eye(6) - J.T @ JT_mp

rng = np.random.default_rng(4)
f_test = rng.standard_normal(3)

print("Dynamic: ||N J^T f|| =", np.linalg.norm(N_dyn @ J.T @ f_test))
print("MP:      ||N J^T f|| =", np.linalg.norm(N_mp @ J.T @ f_test))

Dynamic: ||N J^T f|| = 4.327272826775406e-16
MP:      ||N J^T f|| = 9.842333484766165e-16


## 7. Constraint-consistent joint reference

Double the Shoulder-Lift amplitude to \(1.2\) rad and project the desired joint velocity while contact is active:

$
\dot q^d_c=N\dot q^d,
\qquad
q^d_{k+1}=q^d_k+\dot q^d_c\,\Delta t.
$

This changes the commanded joint trajectory so that it does not directly request motion that violates the active point-contact velocity constraint.

In [9]:
# set use_constraint_consistent_reference=True,


## 8. Change the contact location

The lab suggests moving the allowable contact from `tool0` to the origin of `wrist_3_link`. In RViz, enable TF visualization and observe that constraining the wrist-frame origin does not prevent the distal end-effector from crossing the ground plane.

In [10]:
#  contact_frame_name="wrist_3_link"